# Objetivo
Generar samples y obtener una idea del escalado temporal, todo aprovechando de usar tecnicas de batching programadas; de esta manera generando datos para los modelos de ML clasificativos.

In [25]:
import numpy as np
import pandas as pd
import time
import psutil
import pickle
from scipy.stats import qmc
from lib.oracle import OracleExecutor  # assumes your OracleExecutor is in oracle_wrapper.py


epsilon = 0.1
vev = 246

# Define the ranges for each parameter:
# [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
param_bounds = np.array([
    [130.0, 500.0],    # m_phi (GeV)
    [130.0, 500.0],    # m_A   (GeV)
    [0.95, 1.0],        # sin(b - a)
    [10.0, 10000.0],        # tan(beta)
    [-epsilon, epsilon],# lambda6
    [-epsilon, epsilon],# lambda7
    [0.0, epsilon**4 * vev**2]  # m12^2, agregado para que sea pequeño
])

# Prepare executor
executor = OracleExecutor(nthreads=4)

# Storage for performance metrics
perf_records = []

print("max m12^2:", epsilon**4 * vev**2)


max m12^2: 6.051600000000001


# Runs

In [28]:
import os
import glob
import pickle
import psutil
import time
import numpy as np
from scipy.stats import qmc

batch_size = 15_000
outdir = "data_batches"
max_merge_size_mb = 30

os.makedirs(outdir, exist_ok=True)

In [29]:
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
batch_idx

10

In [33]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



13
Batch 13 saved (15000 points) in 802.5s → data_batches/batch_13.pkl


In [34]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



14
Batch 14 saved (15000 points) in 810.5s → data_batches/batch_14.pkl


In [35]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



15
Batch 15 saved (15000 points) in 811.3s → data_batches/batch_15.pkl



# Testing Speed

In [ ]:
# Define the sampling sizes
sample_sizes = [1, 10, 100, 1_000, 10_000]

for n in sample_sizes:
    # Generate Latin Hypercube samples in [0,1]^7, then scale
    sampler = qmc.LatinHypercube(d=7)
    sample_unit = sampler.random(n)
    param_list = qmc.scale(sample_unit, param_bounds[:,0], param_bounds[:,1])
    
    # Measure memory before run
    process = psutil.Process()
    mem_before = process.memory_info().rss
    
    # Run and time
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=False)
    t1 = time.perf_counter()
    
    mem_after = process.memory_info().rss
    delta_mem = (mem_after - mem_before) / (1024**2)  # in MB
    
    # Save raw results for this batch
    with open(f"oracle_results_{n}.pkl", "wb") as f:
        pickle.dump(results, f)
    
    # Record performance
    perf_records.append({
        "n_points": n,
        "time_sec": t1 - t0,
        "mem_delta_MB": delta_mem
    })
    print(f"Completed batch {n}: time={t1-t0:.2f}s, memory Δ={delta_mem:.1f}MB")

# Save performance table
df_perf = pd.DataFrame(perf_records)
df_perf.to_csv("performance_scaling.csv", index=False)



# Merging

In [11]:
# ------------------------
# Merge old batches if they exceed size threshold
# ------------------------
def merge_batches(folder, prefix="batch_", merged_prefix="merged_", max_size_mb=30):
    files = sorted(glob.glob(f"{folder}/{prefix}*.pkl"))
    group, acc_size, merge_idx = [], 0, 1

    for fp in files:
        fsize = os.path.getsize(fp)
        if (acc_size + fsize) / (1024**2) > max_size_mb:
            # write out merged group
            merged_data = []
            for gfp in group:
                with open(gfp, "rb") as gf:
                    merged_data.append(pickle.load(gf))
                os.remove(gfp)
            mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
            with open(mout, "wb") as mf:
                pickle.dump(merged_data, mf)
            print(f"Merged {len(group)} batches into {mout}")
            merge_idx += 1
            group, acc_size = [], 0

        group.append(fp)
        acc_size += fsize

    # leftover
    if group:
        merged_data = []
        for gfp in group:
            with open(gfp, "rb") as gf:
                merged_data.append(pickle.load(gf))
            os.remove(gfp)
        mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
        with open(mout, "wb") as mf:
            pickle.dump(merged_data, mf)
        print(f"Merged {len(group)} batches into {mout}")

# call it
merge_batches(outdir, max_size_mb=max_merge_size_mb)

Merged 10 batches into data_batches/merged_1.pkl
Merged 2 batches into data_batches/merged_2.pkl
